### KPMP TAL Subset Analysis Notebook

We will filter the KPMP TAL dataset to focus on specific subclasses and visualize some results. This is an improvement over the previous workflow, where we will now independently generate a new UMAP embedding for the TAL subset.

In [ ]:
from pathlib import Path

import re
import requests
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from sklearn.neighbors import NearestNeighbors
from kneed import KneeLocator

In [ ]:
def download(url: str, path: Path) -> None:
    """
    Optimized version from https://scanpy.readthedocs.io/en/stable/tutorials/experimental/dask.html,
    now with better chunk size, connection handling, and progress updates.
    """
    from tqdm.autonotebook import tqdm

    temp_path = path.with_suffix(path.suffix + ".tmp")

    try:
        with requests.Session() as session:
            session.headers.update(
                {"User-Agent": "Mozilla/5.0 (compatible; Python-requests)"}
            )

            response = session.get(url, stream=True, timeout=30)
            response.raise_for_status()

            total_size = int(response.headers.get("content-length", 0))
            chunk_size = 65536  # 64kb chunks

            with tqdm(
                total=total_size,
                unit="B",
                unit_scale=True,
                unit_divisor=1024,
                desc=path.name,
            ) as pbar:

                with open(temp_path, "wb") as f:
                    downloaded = 0
                    for chunk in response.iter_content(chunk_size=chunk_size):
                        if chunk:  # filter out keep-alive chunks
                            size = f.write(chunk)
                            downloaded += size
                            pbar.update(size)

        temp_path.replace(path)

    except Exception as e:
        temp_path.unlink(missing_ok=True)
        raise RuntimeError(f"Failed to download {url}: {e}") from e

### Data citation

**Data source:**
CZI CellxGene Consortium. (2023). *Integrated Single-nucleus and Single-cell RNA-seq of the Adult Human Kidney* [Dataset]. CZ CellxGene. Retrieved from https://cellxgene.cziscience.com/collections/bcb61471-2a44-4d00-a0af-ff085512674c

In [ ]:
kpmp_atlas_path = Path("data/kpmp.h5ad")
kpmp_atlas_path.parent.mkdir(exist_ok=True)
if not kpmp_atlas_path.exists():
    download(
        "https://datasets.cellxgene.cziscience.com/45a06603-f923-45af-b4c3-8ead77aa2e78.h5ad",
        kpmp_atlas_path,
    )

In [ ]:
adata = sc.read(kpmp_atlas_path)
adata

### Data processing

We compute ±3 MAD thresholds on library complexity and mitochondrial/ERCC content, then filter out outlier cells.

In [ ]:
sc.pl.violin(
    adata,
    ["nFeature_RNA", "nCount_RNA", "percent.mt", "percent.er"],
    jitter=0.4,
    multi_panel=True,
)
sc.pl.scatter(adata, x="nCount_RNA", y="percent.mt")
sc.pl.scatter(adata, x="nCount_RNA", y="nFeature_RNA")

In [ ]:
metrics = ["nFeature_RNA", "nCount_RNA", "percent.mt", "percent.er"]
k = 3

thresholds = {}
for m in metrics:
    x = adata.obs[m].values
    med = np.median(x)
    mad = np.median(np.abs(x - med))
    lower = max(0, med - k * mad) # ensure lower limit is non-negative
    upper = med + k * mad
    thresholds[m] = (lower, upper)
    print(f"{m}: median={med:.1f}, MAD={mad:.1f} → keep [{lower:.1f}, {upper:.1f}]")

In [ ]:
filt = np.ones(adata.n_obs, dtype=bool)
for m, (low, high) in thresholds.items():
    filt &= (adata.obs[m] >= low) & (adata.obs[m] <= high)

In [ ]:
adata_qc = adata[filt].copy()
print(f"Kept {adata_qc.n_obs} / {adata.n_obs} cells after MAD-based QC")

In [ ]:
sc.pl.violin(
    adata_qc,
    ["nFeature_RNA", "nCount_RNA", "percent.mt", "percent.er"],
    jitter=0.4,
    multi_panel=True,
)
sc.pl.scatter(adata_qc, x="nCount_RNA", y="percent.mt")
sc.pl.scatter(adata_qc, x="nCount_RNA", y="nFeature_RNA")

### Subsetting TAL Cells

We subset to Thick Ascending Limb (TAL) epithelial cells and inspect the resulting cell count.

In [ ]:
adata_tal = adata_qc[
    adata_qc.obs["cell_type"]
    == "kidney loop of Henle thick ascending limb epithelial cell",
    :,
].copy()
print("TAL subset:", adata_tal.n_obs, "cells")

### Preprocessing v2

We then recompute QC metrics for TAL, normalize, log-transform, regress out covariates, and scale.

In [ ]:
sc.pp.calculate_qc_metrics(adata_tal)

In [ ]:
sc.pp.normalize_total(adata_tal, target_sum=1e4)
sc.pp.log1p(adata_tal)

In [ ]:
sc.pp.regress_out(adata_tal, keys=["nCount_RNA", "percent.mt", "S.Score", "G2M.Score"])
sc.pp.scale(adata_tal, max_value=10)

### PCA and UMAP

We perform PCA, plot cumulative variance, and select the optimal number of PCs via the kneedle algorithm. We then compute mean k-NN distance over a range of k values, plot the curve, and pick the optimal neighborhood size.

We build the neighborhood graph using the parameters selected above, and we compute UMAP using data-driven min_dist. Finally, we save the processed TAL AnnData.

In [ ]:
sc.tl.pca(adata_tal, n_comps=50, svd_solver='arpack')

In [ ]:
evr = np.array(adata_tal.uns["pca"]["variance_ratio"])
cumvar = np.cumsum(evr)

pc_indices = np.arange(1, len(cumvar) + 1)

plt.figure(figsize=(8, 5))
plt.plot(pc_indices, cumvar, marker='o', linestyle='-', color='blue')
plt.xlabel("Principal Component Index")
plt.ylabel("Cumulative Variance Explained")
plt.title("Cumulative Variance Explained by PCA Components")
plt.grid()
plt.show()

In [ ]:
knee_pc = KneeLocator(
    pc_indices,
    cumvar,
    curve="concave",
    direction="increasing",
).knee

n_pcs = knee_pc if knee_pc is not None else int(np.searchsorted(cumvar, 0.90)) + 1
print(f"Selected {n_pcs} PCs (knee at PC {knee_pc})")

In [ ]:
max_k = min(int(0.01 * adata_tal.n_obs), 200)
ks = np.arange(5, max_k + 1)

mean_kd = []
for k in ks:
    nbrs = NearestNeighbors(n_neighbors=k + 1).fit(adata_tal.obsm["X_pca"][:, :n_pcs])
    dists, _ = nbrs.kneighbors(adata_tal.obsm["X_pca"][:, :n_pcs])
    mean_kd.append(dists[:, -1].mean())

plt.figure(figsize=(8, 5))
plt.plot(ks, mean_kd, marker='o', linestyle='-', color='blue')
plt.xlabel("k (number of neighbors)")
plt.ylabel("Mean k-NN Distance")
plt.title("Mean k-NN Distance vs. k")
plt.grid()
plt.show()

In [ ]:
knee_k = KneeLocator(
    ks,
    mean_kd,
    curve="concave",
    direction="increasing",
).knee

n_neighbors = knee_k if knee_k is not None else max(10, int(0.01 * adata_tal.n_obs))
print(f"Selected n_neighbors = {n_neighbors} (knee at rank {knee_k})")

In [ ]:
sc.pp.neighbors(adata_tal, n_pcs=n_pcs, n_neighbors=n_neighbors)

In [ ]:
median_knn_dist = np.median(dists[:, k - 1])
min_dist = median_knn_dist / np.max(dists)
print(f"Median k-NN distance: {median_knn_dist:.4f}, min_dist = {min_dist:.4f}")

In [ ]:
sc.tl.umap(adata_tal, n_components=2, min_dist=min_dist, spread=1.0, random_state=42)

In [ ]:
adata_tal_path = Path("data/kpmp_tal.h5ad")
adata_tal_path.parent.mkdir(exist_ok=True)
adata_tal.write(adata_tal_path)

### Data visualization

Plot UMAPs colored by key annotations in `obs` (e.g. `subclass.l2`, `disease`, `tissue`) and save the figures.

In [ ]:
for color in ["subclass.l2", "disease", "tissue"]:
    if color in adata_tal.obs:
        sc.pl.umap(
            adata_tal,
            color=[color],
            frameon=False,
            wspace=0.35,
            save=f"_kpmp_tal_umap_{color}.png",
        )

In [ ]:
def stage_to_number(label):
    m = re.match(r"(\d+)-year-old stage", label)
    if m:
        return int(m.group(1))

    m = re.match(r"(first|second|third|fourth|fifth|sixth|seventh|eighth) decade stage", label)
    if m:
        words = ["first","second","third","fourth","fifth","sixth","seventh","eighth"]
        d = words.index(m.group(1)) + 1
        return 10*d - 5
    raise ValueError(f"Unknown stage label: {label}")

In [ ]:
development_stages = adata_tal.obs["development_stage"].unique()
stage_numbers = np.array([stage_to_number(s) for s in development_stages])

df = pd.DataFrame({"label": development_stages, "age": stage_numbers})
df["age"] = df["age"].astype(int)
df = df.sort_values("age")

In [ ]:
orig = mpl.colormaps["Reds"]
colors_trunc = orig(np.linspace(0.5, 1.0, 256))
cmap = mpl.colors.LinearSegmentedColormap.from_list("Reds_trunc", colors_trunc)

norm = mpl.colors.Normalize(vmin=df.age.min(), vmax=df.age.max())

age_lookup = dict(zip(df["label"], df["age"]))
cell_ages = adata_tal.obs["development_stage"].map(age_lookup).values
cell_colors = cmap(norm(cell_ages))

In [ ]:
umap = adata_tal.obsm["X_umap"]

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(
    umap[:, 0], umap[:, 1], c=cell_colors, s=1, linewidth=0, rasterized=True
)
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.set_title("TAL cells colored by development stage")

sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array(df["age"])
cbar = fig.colorbar(sm, ax=ax, orientation="vertical", fraction=0.04, pad=0.04)
cbar.set_label("Implied age (years)")

plt.tight_layout()
fig.savefig("figures/kpmp_tal_development_stage_umap.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
mapping = {"C-TAL": "C-TAL", "M-TAL": "M-TAL", "aTAL1": "aTAL", "aTAL2": "aTAL"}
merged = adata_tal.obs["subclass.l2"].astype(str).map(mapping)

In [ ]:
adata_tal.obs["merged_subclass"] = pd.Categorical(
    merged, categories=["C-TAL", "M-TAL", "aTAL"], ordered=True
)
adata_filtered = adata_tal[adata_tal.obs["merged_subclass"].notna(), :].copy()

In [ ]:
adata_tal_filtered_path = Path("data/kpmp_tal_filtered.h5ad")
adata_tal_filtered_path.parent.mkdir(exist_ok=True)
adata_filtered.write(adata_tal_filtered_path)

In [ ]:
sc.pl.umap(
    adata_filtered,
    color="merged_subclass",
    title="TAL subclasses (C-TAL, M-TAL, aTAL)",
    frameon=False,
    save="_kpmp_tal_umap_merged_subclass.png",
)